# 03 — Akash Deployment and Pull Validation

Curated starter notebook for post-deployment artifact validation.

This notebook validates pulled Akash `run.json` files and compares final metrics for oracle/headline/degraded regimes.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RUN_FILES = {
    "oracle": ROOT / "results/akash_2c_oracle/run.json",
    "headline": ROOT / "results/akash_2c_headline_dseq26607956/run.json",
    "degraded": ROOT / "results/akash_2c_degraded_dseq26609636/run.json",
}


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

In [ ]:
required_config_keys = {"defense", "attack_type", "model_size", "n_rounds", "rep_weights"}
required_round_keys = {"round", "perplexity", "round_time", "agg_time", "asr", "asr_per_position", "asr_best_position"}

validation_rows = []
runs = {}
for name, path in RUN_FILES.items():
    exists = path.exists()
    if not exists:
        validation_rows.append({"run": name, "path": str(path), "exists": False, "valid": False, "reason": "missing file"})
        continue

    payload = load_json(path)
    runs[name] = payload

    config = payload.get("config", {})
    rounds = payload.get("rounds", [])
    if not rounds:
        validation_rows.append({"run": name, "path": str(path), "exists": True, "valid": False, "reason": "no rounds logged"})
        continue

    last = rounds[-1]
    missing_cfg = sorted(required_config_keys - set(config.keys()))
    missing_round = sorted(required_round_keys - set(last.keys()))
    valid = not missing_cfg and not missing_round
    reason = "ok" if valid else f"missing config={missing_cfg}, missing round={missing_round}"

    validation_rows.append({"run": name, "path": str(path), "exists": True, "valid": valid, "reason": reason})

validation_rows

In [ ]:
# Display validation status.
validation_rows

In [ ]:
# Compare final metrics across validated runs.
final_rows = []
for name, run in runs.items():
    last = run["rounds"][-1]
    final_rows.append({
        "run": name,
        "perplexity": float(last["perplexity"]),
        "asr": float(last.get("asr", 0.0)),
        "agg_time": float(last["agg_time"]),
        "round_time": float(last["round_time"]),
        "best_position": int(last.get("asr_best_position", 0)),
    })

final_rows

In [ ]:
labels = [row["run"] for row in final_rows]
ppl = [row["perplexity"] for row in final_rows]
asr = [row["asr"] for row in final_rows]
agg = [row["agg_time"] for row in final_rows]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(labels, ppl, color="#4C78A8")
axes[0].set_title("Final Perplexity")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(labels, asr, color="#E45756")
axes[1].set_title("Final ASR")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(axis="y", alpha=0.3)

axes[2].bar(labels, agg, color="#72B7B2")
axes[2].set_title("Final Aggregation Time")
axes[2].tick_params(axis="x", rotation=30)
axes[2].grid(axis="y", alpha=0.3)

fig.suptitle("Akash Pull Validation Snapshot")
fig.tight_layout()
plt.show()

## Conclusion

- This notebook verifies that pulled Akash artifacts have the expected schema and comparable final metrics.
- Use it as a first gate before generating publication plots or writing run summaries.